In [ ]:
# import pandas as pd
# import numpy as np
# import os
# from sklearn.cluster import KMeans
# from sklearn.preprocessing import StandardScaler
# from datetime import timedelta

# file_path = r'second_stage_inputs\G11\dsas_g11_generator_bearings_output.xlsx'
# output_filename = r'outputs\G11\dsas_g11_generator_bearings_clustering\clustering\dsas_g11_generator_bearings\clustering_g11_generator_bearings_output4.xlsx'
# target_sensors = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']

# def run_smart_analysis():
#     if not os.path.exists(file_path):
#         print(f"❌ File not found: {file_path}")
#         return

#     # 1. Data Loading & Preprocessing
#     df = pd.read_excel(file_path)
    
#     if 'date' in df.columns:
#         df['date'] = pd.to_datetime(df['date'])
    
#     df_model = df.dropna(subset=target_sensors).copy()
    
#     # 2. Denoising (Moving Average to eliminate operator/sensor glitches)
#     # This step ensures we analyze "Real Trends" rather than "Outliers"
#     for sensor in target_sensors:
#         df_model[f'{sensor}_smooth'] = df_model[sensor].rolling(window=5, center=True).mean()
    
#     df_model = df_model.dropna(subset=[f'{s}_smooth' for s in target_sensors]).copy()
#     smooth_cols = [f'{s}_smooth' for s in target_sensors]

#     # 3. Feature Scaling
#     scaler = StandardScaler()
#     scaled_data = scaler.fit_transform(df_model[smooth_cols])
    
#     # 4. Clustering (Behavioral Segmentation)
#     kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
#     df_model['Behavior_Cluster'] = kmeans.fit_predict(scaled_data)

#     # 5. Engineering Health Metrics (Degradation Index)
#     # Calculating Euclidean distance from the cluster centroid
#     distances = np.linalg.norm(scaled_data - kmeans.cluster_centers_[df_model['Behavior_Cluster']], axis=1)
#     df_model['Degradation_Index'] = distances 

#     # 6. Standardized Health Labeling (English Industry Standards)
#     def get_health_status(row):
#         if row['Degradation_Index'] > 2.5:
#             return "Investigation Needed (Operational Drift)"
#         elif row['Degradation_Index'] > 1.5:
#             return "Observation Required (Pattern Change)"
#         else:
#             return "Healthy (Optimal Performance)"

#     df_model['Health_Status'] = df_model.apply(get_health_status, axis=1)

#     # 7. Filtering for the Last 30 Days
#     if 'date' in df_model.columns:
#         last_date = df_model['date'].max()
#         one_month_ago = last_date - timedelta(days=30)
#         final_output = df_model[df_model['date'] >= one_month_ago].copy()
#     else:
#         final_output = df_model

#     # 8. Exporting to Excel
#     try:
#         os.makedirs(os.path.dirname(output_filename), exist_ok=True)
#         final_output.to_excel(output_filename, index=False)
#         print("🚀 Analysis completed successfully.")
#         print(f"📊 Labels updated to International Engineering Standards.")
#         print(f"📁 Output saved: {output_filename}")
#     except Exception as e:
#         print(f"❌ Error saving file: {e}")

# if __name__ == "__main__":
#     run_smart_analysis()

In [ ]:
import pandas as pd
import numpy as np
import os
import time
from datetime import datetime, timedelta
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# غیرفعال کردن هشدارهای غیرضروری
import warnings
warnings.filterwarnings('ignore')

def run_smart_analysis():
    """اجرای تحلیل با K-Means Clustering برای ژنراتور و ذخیره خروجی"""
    
    print("="*60)
    print(f"🔄 شروع تحلیل در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*60)
    
    # --- Configuration ---
    file_path = r'second_stage_inputs\G11\dsas_g11_generator_bearings_output.xlsx'
    output_filename = r'outputs\G11\dsas_g11_generator_bearings_clustering\clustering\dsas_g11_generator_bearings\clustering_g11_generator_bearings_output4.xlsx'
    
    target_sensors = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 
                      'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']

    if not os.path.exists(file_path):
        print(f"❌ File not found: {file_path}")
        return None

    # 1. Data Loading & Preprocessing
    print("🔄 مرحله 1: بارگذاری و پیش‌پردازش داده...")
    df = pd.read_excel(file_path)
    
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])
    
    df_model = df.dropna(subset=target_sensors).copy()
    print(f"✅ داده بارگذاری شد. تعداد رکوردها: {len(df_model):,}")
    
    # 2. Denoising (Moving Average)
    print("🔄 مرحله 2: حذف نویز با میانگین متحرک...")
    for sensor in target_sensors:
        df_model[f'{sensor}_smooth'] = df_model[sensor].rolling(window=5, center=True).mean()
    
    smooth_cols = [f'{s}_smooth' for s in target_sensors]
    df_model = df_model.dropna(subset=smooth_cols).copy()
    print(f"✅ پس از حذف نویز: {len(df_model):,} رکورد")

    # 3. Feature Scaling
    print("🔄 مرحله 3: استانداردسازی داده‌ها...")
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_model[smooth_cols])
    print(f"✅ استانداردسازی با {len(smooth_cols)} سنسور انجام شد")
    
    # 4. Clustering (Behavioral Segmentation)
    print("🔄 مرحله 4: خوشه‌بندی با K-Means...")
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    df_model['Behavior_Cluster'] = kmeans.fit_predict(scaled_data)
    
    # آمار خوشه‌ها
    cluster_counts = df_model['Behavior_Cluster'].value_counts().sort_index()
    print(f"   تعداد خوشه‌ها: {len(cluster_counts)}")
    for cluster_id, count in cluster_counts.items():
        print(f"   خوشه {cluster_id}: {count:,} رکورد ({count/len(df_model)*100:.2f}%)")

    # 5. Engineering Health Metrics (Degradation Index)
    print("🔄 مرحله 5: محاسبه شاخص تخریب...")
    distances = np.linalg.norm(scaled_data - kmeans.cluster_centers_[df_model['Behavior_Cluster']], axis=1)
    df_model['Degradation_Index'] = distances 
    print(f"   محدوده شاخص تخریب: {df_model['Degradation_Index'].min():.4f} تا {df_model['Degradation_Index'].max():.4f}")

    # 6. Standardized Health Labeling
    print("🔄 مرحله 6: لیبل‌گذاری وضعیت سلامت...")
    
    def get_health_status(row):
        if row['Degradation_Index'] > 2.5:
            return "Investigation Needed (Operational Drift)"
        elif row['Degradation_Index'] > 1.5:
            return "Observation Required (Pattern Change)"
        else:
            return "Healthy (Optimal Performance)"

    df_model['Health_Status'] = df_model.apply(get_health_status, axis=1)
    
    # نمایش توزیع وضعیت‌ها
    status_counts = df_model['Health_Status'].value_counts()
    print(f"\n📊 توزیع وضعیت‌ها:")
    for status, count in status_counts.items():
        print(f"   {status}: {count:,} ({count/len(df_model)*100:.2f}%)")

    # 7. Filtering for the Last 30 Days
    print("🔄 مرحله 7: فیلتر کردن داده‌های ۳۰ روز آخر...")
    if 'date' in df_model.columns:
        last_date = df_model['date'].max()
        one_month_ago = last_date - timedelta(days=30)
        final_output = df_model[df_model['date'] >= one_month_ago].copy()
        print(f"   بازه خروجی: {one_month_ago} تا {last_date}")
        print(f"   تعداد رکوردهای ۳۰ روز آخر: {len(final_output):,}")
    else:
        final_output = df_model
        print("   ⚠️ ستون 'date' وجود ندارد، تمام داده‌ها ذخیره می‌شوند.")

    # نمایش آمار نهایی
    print(f"\n📊 آمار نهایی:")
    print(f"   کل رکوردها: {len(df_model):,}")
    print(f"   رکوردهای ۳۰ روز آخر: {len(final_output):,}")
    
    # نمایش نمونه‌هایی که نیاز به بررسی دارند
    investigation_needed = final_output[final_output['Health_Status'].str.contains('Investigation Needed')]
    if len(investigation_needed) > 0:
        print(f"\n⚠️ تعداد رکوردهای نیازمند بررسی: {len(investigation_needed):,}")
        print(f"   درصد نیازمند بررسی: {len(investigation_needed)/len(final_output)*100:.2f}%")

    # 8. Exporting to Excel
    print("💾 مرحله 8: ذخیره خروجی...")
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        final_output.to_excel(output_filename, index=False)
        print(f"✅ Analysis completed successfully.")
        print(f"📊 Labels updated to International Engineering Standards.")
        print(f"📁 Output saved: {output_filename}")
        print(f"📊 تعداد رکوردهای نهایی: {len(final_output):,}")
        print(f"📋 تعداد ستون‌ها: {len(final_output.columns)}")
    except Exception as e:
        print(f"❌ Error saving file: {e}")
        return None
    
    print("="*60)
    print(f"✅ تحلیل در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} کامل شد")
    print("="*60)
    
    return final_output

def run_scheduler():
    """
    بررسی مداوم برای اجرا در زمان‌های مشخص (هر روز)
    """
    print("="*60)
    print("🔄 برنامه زمان‌بندی خودکار شروع به کار کرد - تحلیل خوشه‌بندی ژنراتور با K-Means")
    print("="*60)
    print("⏰ زمان‌های اجرا (هر روز):")
    print("   - ساعت 10:00")
    print("   - ساعت 10:05")
    print("   - ساعت 10:10")
    print("="*60)
    print("💡 برای توقف برنامه، Ctrl+C را بزنید")
    print("="*60)
    
    last_run_time = None  # فقط برای جلوگیری از اجرای مجدد در یک زمان
    
    while True:
        try:
            now = datetime.now()
            current_time = now.strftime("%H:%M")
            
            # بررسی زمان‌های مشخص
            if current_time in ["22:06", "22:08", "22:10"]:
                # فقط چک می‌کنیم که در همین زمان دوبار اجرا نشود
                if last_run_time != current_time:
                    print("\n" + "="*60)
                    print(f"⏰ زمان اجرا فرا رسید: {now.strftime('%Y-%m-%d %H:%M:%S')}")
                    print("="*60)
                    
                    # اجرای تابع اصلی
                    result = run_smart_analysis()
                    
                    if result is not None:
                        print("\n" + "="*60)
                        print("✅ اجرای زمان‌بندی شده با موفقیت کامل شد!")
                        print("="*60)
                    else:
                        print("\n" + "="*60)
                        print("❌ اجرای زمان‌بندی شده با شکست مواجه شد!")
                        print("="*60)
                    
                    # ثبت زمان اجرا
                    last_run_time = current_time
                    
                    # 10 ثانیه صبر کن تا از اجرای مجدد در همان دقیقه جلوگیری شود
                    time.sleep(10)
            
            # هر 10 ثانیه یکبار بررسی کن
            time.sleep(10)
            
        except KeyboardInterrupt:
            print("\n" + "="*60)
            print("⏹️ برنامه با دستور کاربر متوقف شد")
            print(f"⏹️ زمان توقف: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print("="*60)
            break
            
        except Exception as e:
            print(f"❌ خطا در حلقه اصلی: {e}")
            print("🔄 ادامه اجرا...")
            time.sleep(60)

# اجرای اصلی
if __name__ == "__main__":
    try:
        print("="*60)
        print("🚀 شروع برنامه تحلیل خوشه‌بندی ژنراتور با K-Means")
        print("="*60)
        
        # شروع زمان‌بندی
        run_scheduler()
        
    except Exception as e:
        print(f"❌ خطای غیرمنتظره: {e}")
        input("برای خروج Enter بزنید...")